In [2]:

## QEMAR DATOS E INFORME DE EXCEL DE PROCESAMIENTO

# ============================================================
# GENERADOR DE RASTER HISTÓRICOS DE CULTIVOS
# ============================================================
#
# FUENTES:
#
#   1. Excel histórico
#   2. Shapefiles históricos
#
# LLAVE:
#
#   CODSIA
#
# RESULTADOS:
#
#   2022_EDAD.tif
#   2022_AREA_BRUTA.tif
#   2024_EDAD.tif
#   2024_AREA_BRUTA.tif
#   ...
#
# CONTROL:
#
#   CONTROL_EXCEL.csv
#
#
# IMPORTANTE:
#
# El proceso solamente trabaja con los años definidos
# en ANOS.
#
# Ejemplo:
#
#     ANOS = [2022, 2024]
#
# El control solamente tendrá registros de:
#
#     2022
#     2024
#
# ============================================================


# ============================================================
# IMPORTACIONES
# ============================================================

import os
import re

import pandas as pd
import geopandas as gpd
import numpy as np

import rasterio

from rasterio.transform import from_origin

from rasterio.features import rasterize


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================


# ------------------------------------------------------------
# CARPETA PRINCIPAL
# ------------------------------------------------------------

CARPETA_DATOS = (
    r"C:\Users\jjruiz\OneDrive - Sector Agro\Geodata\Analisis datos a raster\TEMP\depurados"
)


# ------------------------------------------------------------
# ARCHIVO EXCEL
# ------------------------------------------------------------

ARCHIVO_EXCEL = os.path.join(
    CARPETA_DATOS,
    "Reporte_Biosal.xlsx"
)


# ------------------------------------------------------------
# CARPETA DE SALIDA
# ------------------------------------------------------------

CARPETA_SALIDA = os.path.join(
    CARPETA_DATOS,
    "RASTER_HISTORICOS"
)


os.makedirs(
    CARPETA_SALIDA,
    exist_ok=True
)


# ------------------------------------------------------------
# AÑOS A PROCESAR
#
# EJEMPLOS:
#
# ANOS = [2020]
#
# ANOS = [2022, 2024]
#
# ANOS = [2019, 2020, 2021, 2022]
#
# TODOS:
#
# ANOS = list(range(2014, 2026))
# ------------------------------------------------------------

#ANOS = [2124,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
ANOS = [2024]



# ------------------------------------------------------------
# ATRIBUTOS A RASTERIZAR
#
# Deben existir exactamente con estos nombres en el Excel.
# ------------------------------------------------------------

ATRIBUTOS = ["Variedad_TIFF","Corte","Edad","Rdto","TAH","TAHM","%Sac.Cana","Num.Riegos","Vejez","Semanas mad.","Luvias","Lluvias (0 - 3 Meses)","Lluvias 10d Ant.Cos","Ton.Azucar"]
#ATRIBUTOS = ["Variedad_TIFF"]


# ------------------------------------------------------------
# RESOLUCIÓN DEL RASTER
#
# 10 = 10 x 10 metros
# ------------------------------------------------------------

RESOLUCION = 10


# ------------------------------------------------------------
# CRS DE SALIDA
# ------------------------------------------------------------

CRS_SALIDA = "EPSG:21896"


# ------------------------------------------------------------
# VALOR NODATA
# ------------------------------------------------------------

NODATA = -9999


# ============================================================
# 2. NORMALIZAR CODSIA
# ============================================================
#
# ESTA FUNCIÓN ES CRÍTICA.
#
# CODSIA NO ES UN NÚMERO.
#
# Es una llave alfanumérica.
#
# Puede tener:
#
#   9 caracteres:
#
#       010001008
#
#   10 caracteres:
#
#       010001009B
#
#
# Si Excel lo convirtió accidentalmente a número:
#
#       10001008
#
# se recupera:
#
#       010001008
#
#
# Si tiene letra:
#
#       10001009B
#
# se recupera:
#
#       010001009B
#
#
# NUNCA se eliminan ceros.
# NUNCA se convierte a int.
# NUNCA se convierte a float.
#
# ============================================================


def normalizar_codsia(valor):

    """
    Normaliza CODSIA conservando su estructura original.

    Reglas:

    - CODSIA numérico:
        10001008
        ->
        010001008

    - CODSIA numérico ya correcto:
        010001008
        ->
        010001008

    - CODSIA con letra:
        10001009B
        ->
        010001009B

    - CODSIA con letra ya correcto:
        010001009B
        ->
        010001009B

    - Cualquier otro formato no se modifica.
    """

    # --------------------------------------------------------
    # Valor vacío
    # --------------------------------------------------------

    if pd.isna(valor):

        return None


    # --------------------------------------------------------
    # Convertir a texto
    #
    # IMPORTANTE:
    #
    # Desde aquí nunca se trata como número.
    # --------------------------------------------------------

    texto = str(
        valor
    ).strip()


    # --------------------------------------------------------
    # Eliminar espacios externos
    # --------------------------------------------------------

    texto = texto.strip()


    # --------------------------------------------------------
    # Caso generado por Excel:
    #
    # 10001008.0
    #
    # Lo convertimos a:
    #
    # 10001008
    # --------------------------------------------------------

    if texto.endswith(".0"):

        texto = texto[:-2]


    # ========================================================
    # CASO 1
    #
    # SOLO NÚMEROS
    #
    # ========================================================

    if texto.isdigit():

        # ----------------------------------------------------
        # Siempre completar hasta 9 dígitos.
        #
        # Si ya tiene 9:
        #
        # 010001008
        #
        # queda igual.
        #
        # Si tiene 8:
        #
        # 10001008
        #
        # queda:
        #
        # 010001008
        # ----------------------------------------------------

        texto = texto.zfill(
            9
        )


        return texto


    # ========================================================
    # CASO 2
    #
    # NÚMEROS + UNA LETRA AL FINAL
    #
    # Ejemplo:
    #
    # 10001009B
    #
    # ========================================================

    match = re.fullmatch(

        r"(\d+)([A-Za-z])",

        texto

    )


    if match:

        parte_numerica = match.group(
            1
        )

        letra = match.group(
            2
        )


        # ----------------------------------------------------
        # Completar la parte numérica a 9 dígitos.
        # ----------------------------------------------------

        parte_numerica = (
            parte_numerica.zfill(
                9
            )
        )


        # ----------------------------------------------------
        # Conservar la letra.
        # ----------------------------------------------------

        return (

            parte_numerica

            +

            letra

        )


    # ========================================================
    # OTROS FORMATOS
    #
    # No modificar.
    #
    # Esto evita alterar accidentalmente códigos especiales.
    # ========================================================

    return texto


# ============================================================
# 3. LIMPIAR NOMBRE DE ATRIBUTO
# ============================================================


def limpiar_nombre_archivo(nombre):

    nombre = str(
        nombre
    ).strip()


    # --------------------------------------------------------
    # Espacios por _
    # --------------------------------------------------------

    nombre = nombre.replace(
        " ",
        "_"
    )


    # --------------------------------------------------------
    # Eliminar caracteres especiales
    # --------------------------------------------------------

    nombre = re.sub(

        r"[^A-Za-z0-9_\-]",

        "",

        nombre

    )


    return nombre.upper()


# ============================================================
# 4. OBTENER RUTA DEL SHAPEFILE
# ============================================================


def obtener_ruta_shp(nombre):

    nombre = str(
        nombre
    ).strip()


    # --------------------------------------------------------
    # Agregar extensión si no existe
    # --------------------------------------------------------

    if not nombre.lower().endswith(
        ".shp"
    ):

        nombre += ".shp"


    return os.path.join(

        CARPETA_DATOS,

        nombre

    )


# ============================================================
# 5. CONVERTIR ATRIBUTO A NUMÉRICO
# ============================================================


def convertir_a_numerico(serie):

    # --------------------------------------------------------
    # Si ya es numérico
    # --------------------------------------------------------

    if pd.api.types.is_numeric_dtype(
        serie
    ):

        return pd.to_numeric(

            serie,

            errors="coerce"

        )


    # --------------------------------------------------------
    # Convertir temporalmente a texto
    # --------------------------------------------------------

    serie = (

        serie

        .astype("string")

        .str.strip()

    )


    # --------------------------------------------------------
    # Coma decimal -> punto
    # --------------------------------------------------------

    serie = serie.str.replace(

        ",",

        ".",

        regex=False

    )


    return pd.to_numeric(

        serie,

        errors="coerce"

    )


# ============================================================
# 6. INICIO
# ============================================================


print("\n")

print(
    "=" * 80
)

print(
    "GENERADOR DE RASTER HISTÓRICOS"
)

print(
    "=" * 80
)


print(
    f"\nAños seleccionados: {ANOS}"
)


print(
    f"Atributos: {ATRIBUTOS}"
)


print(
    f"Resolución: "
    f"{RESOLUCION} x {RESOLUCION} metros"
)


# ============================================================
# 7. VALIDAR ARCHIVO EXCEL
# ============================================================


if not os.path.exists(
    ARCHIVO_EXCEL
):

    raise FileNotFoundError(

        f"\nNo existe el archivo:\n"
        f"{ARCHIVO_EXCEL}"

    )


# ============================================================
# 8. DEFINIR COLUMNAS NECESARIAS
# ============================================================
#
# NO cargamos todas las columnas del Excel.
#
# Solamente:
#
#   CODSIA
#   year
#   Capa_producto_final
#   atributos seleccionados
#
# ============================================================


COLUMNAS_EXCEL = [

    "CODSIA",

    "year",

    "Capa_producto_final"

] + ATRIBUTOS


# ------------------------------------------------------------
# Eliminar nombres repetidos
# ------------------------------------------------------------

COLUMNAS_EXCEL = list(
    dict.fromkeys(
        COLUMNAS_EXCEL
    )
)


# ============================================================
# 9. LEER EXCEL
# ============================================================


print("\n")

print(
    "=" * 80
)

print(
    "LECTURA DEL EXCEL"
)

print(
    "=" * 80
)


print(
    "Leyendo únicamente las columnas necesarias..."
)


df = pd.read_excel(

    ARCHIVO_EXCEL,

    usecols=COLUMNAS_EXCEL,

    dtype={
        "CODSIA": "string"
    }

)


print(
    f"Filas leídas: "
    f"{len(df):,}"
)


# ============================================================
# 10. VALIDAR COLUMNAS
# ============================================================


COLUMNAS_REQUERIDAS = [

    "CODSIA",

    "year",

    "Capa_producto_final"

] + ATRIBUTOS


faltantes = [

    columna

    for columna in COLUMNAS_REQUERIDAS

    if columna not in df.columns

]


if faltantes:

    raise ValueError(

        "\nFaltan columnas en el Excel:\n\n"

        +
        "\n".join(
            faltantes
        )

    )


# ============================================================
# 11. NORMALIZAR EXCEL
# ============================================================


# ------------------------------------------------------------
# CODSIA
# ------------------------------------------------------------

df["CODSIA"] = (

    df["CODSIA"]

    .apply(
        normalizar_codsia
    )

    .astype("string")

)


# ------------------------------------------------------------
# YEAR
# ------------------------------------------------------------

df["year"] = pd.to_numeric(

    df["year"],

    errors="coerce"

)


# ------------------------------------------------------------
# CAPA_PRODUCTO_FINAL
# ------------------------------------------------------------

df["Capa_producto_final"] = (

    df[
        "Capa_producto_final"
    ]

    .astype("string")

    .str.strip()

)


# ============================================================
# 12. FILTRAR INMEDIATAMENTE LOS AÑOS
# ============================================================
#
# ESTA ES UNA DE LAS OPTIMIZACIONES PRINCIPALES.
#
# Después de esta operación solamente se procesan los años
# seleccionados.
#
# ============================================================


df_proceso = df[
    df["year"].isin(
        ANOS
    )
].copy()


# ------------------------------------------------------------
# Ya no necesitamos el Excel completo en memoria.
# ------------------------------------------------------------

del df


# ============================================================
# 13. RESULTADO DEL FILTRO
# ============================================================


print("\n")

print(
    "=" * 80
)

print(
    "DATOS SELECCIONADOS"
)

print(
    "=" * 80
)


print(
    f"Años seleccionados: "
    f"{ANOS}"
)


print(
    f"Registros a procesar: "
    f"{len(df_proceso):,}"
)


# ------------------------------------------------------------
# Si no hay registros
# ------------------------------------------------------------

if df_proceso.empty:

    print(
        "\nNo existen registros "
        "para los años seleccionados."
    )

    raise SystemExit


# ============================================================
# 14. ID DEL REGISTRO
# ============================================================
#
# El ID identifica cada registro dentro de los datos
# realmente procesados.
#
# ============================================================


df_proceso["_ID_EXCEL"] = np.arange(

    1,

    len(df_proceso) + 1

)


# ============================================================
# 15. CREAR CONTROL
# ============================================================
#
# IMPORTANTE:
#
# El control se construye desde df_proceso.
#
# Por lo tanto:
#
# ANOS = [2022, 2024]
#
# significa que el CSV solamente tendrá registros de
# 2022 y 2024.
#
# ============================================================


control = df_proceso[
    [
        "_ID_EXCEL",
        "year",
        "CODSIA",
        "Capa_producto_final"
    ]
].copy()


# ------------------------------------------------------------
# Estado inicial
# ------------------------------------------------------------

control[
    "ESTADO"
] = "PENDIENTE"


# ------------------------------------------------------------
# SHP
# ------------------------------------------------------------

control[
    "SHP_ENCONTRADO"
] = "NO"


# ------------------------------------------------------------
# CODSIA en SHP
# ------------------------------------------------------------

control[
    "CODSIA_EN_SHP"
] = "NO"


# ------------------------------------------------------------
# Geometría
# ------------------------------------------------------------

control[
    "GEOMETRIA"
] = "NO"


# ------------------------------------------------------------
# Estado del valor y cantidad de píxeles por atributo
# ------------------------------------------------------------

for atributo in ATRIBUTOS:

    nombre = limpiar_nombre_archivo(
        atributo
    )


    control[
        f"VALOR_{nombre}"
    ] = "NO"


    control[
        f"PIXEL_{nombre}"
    ] = 0


# ------------------------------------------------------------
# Detalle
# ------------------------------------------------------------

control[
    "DETALLE"
] = ""


# ============================================================
# 16. PROCESAR AÑO POR AÑO
# ============================================================


for anio in ANOS:

    print("\n\n")

    print(
        "#" * 80
    )

    print(
        f"# PROCESANDO {anio}"
    )

    print(
        "#" * 80
    )


    # --------------------------------------------------------
    # Filtrar solamente dentro de df_proceso
    # --------------------------------------------------------

    df_anio = df_proceso[
        df_proceso["year"] == anio
    ].copy()


    print(
        f"Registros: "
        f"{len(df_anio):,}"
    )


    if df_anio.empty:

        print(
            "No existen registros."
        )

        continue


    # ========================================================
    # AGRUPAR POR CAPA
    # ========================================================

    grupos = df_anio.groupby(

        "Capa_producto_final",

        dropna=False

    )


    print(
        f"SHP diferentes: "
        f"{len(grupos):,}"
    )


    # ========================================================
    # LISTA DE GEODATAFRAMES
    # ========================================================

    lista_gdf = []


    # ========================================================
    # PROCESAR CADA SHAPEFILE
    # ========================================================

    for (

        nombre_capa,

        df_capa

    ) in grupos:


        print("\n")

        print(
            "-" * 70
        )

        print(
            f"Procesando: "
            f"{nombre_capa}"
        )

        print(
            "-" * 70
        )


        # ====================================================
        # CAPA VACÍA
        # ====================================================

        if pd.isna(
            nombre_capa
        ):

            indices = df_capa.index


            control.loc[
                indices,
                "ESTADO"
            ] = "CAPA_VACIA"


            control.loc[
                indices,
                "DETALLE"
            ] = (
                "Capa_producto_final vacía"
            )


            continue


        # ====================================================
        # RUTA
        # ====================================================

        ruta_shp = obtener_ruta_shp(

            nombre_capa

        )


        print(
            ruta_shp
        )


        # ====================================================
        # SHP NO ENCONTRADO
        # ====================================================

        if not os.path.exists(
            ruta_shp
        ):

            print(
                "❌ SHP NO ENCONTRADO"
            )


            indices = df_capa.index


            control.loc[
                indices,
                "ESTADO"
            ] = (
                "SHP_NO_ENCONTRADO"
            )


            control.loc[
                indices,
                "DETALLE"
            ] = (
                f"No existe: "
                f"{nombre_capa}"
            )


            continue


        # ====================================================
        # SHP ENCONTRADO
        # ====================================================

        indices = df_capa.index


        control.loc[
            indices,
            "SHP_ENCONTRADO"
        ] = "SI"


        # ====================================================
        # LEER SHP
        # ====================================================

        try:

            gdf = gpd.read_file(

                ruta_shp

            )


        except Exception as e:

            control.loc[
                indices,
                "ESTADO"
            ] = (
                "ERROR_LECTURA_SHP"
            )


            control.loc[
                indices,
                "DETALLE"
            ] = str(e)


            print(
                f"❌ Error: {e}"
            )


            continue


        print(
            f"Polígonos SHP: "
            f"{len(gdf):,}"
        )


        # ====================================================
        # VALIDAR CODSIA
        # ====================================================

        if "CODSIA" not in gdf.columns:

            control.loc[
                indices,
                "ESTADO"
            ] = (
                "SHP_SIN_CODSIA"
            )


            control.loc[
                indices,
                "DETALLE"
            ] = (
                "El SHP no contiene "
                "la columna CODSIA"
            )


            continue


        # ====================================================
        # NORMALIZAR CODSIA DEL SHP
        # ====================================================
        #
        # EXACTAMENTE LA MISMA FUNCIÓN QUE UTILIZAMOS
        # PARA EL EXCEL.
        #
        # ====================================================

        gdf["CODSIA"] = (

            gdf["CODSIA"]

            .apply(
                normalizar_codsia
            )

            .astype("string")

        )


        # ====================================================
        # VALIDAR CRS
        # ====================================================

        if gdf.crs is None:

            control.loc[
                indices,
                "ESTADO"
            ] = (
                "SHP_SIN_CRS"
            )


            control.loc[
                indices,
                "DETALLE"
            ] = (
                "El SHP no tiene "
                "CRS definido"
            )


            continue


        # ====================================================
        # REPROYECTAR
        # ====================================================

        if gdf.crs != CRS_SALIDA:

            gdf = gdf.to_crs(

                CRS_SALIDA

            )


        # ====================================================
        # CODSIA DISPONIBLES EN EL SHP
        # ====================================================

        codsias_shp = set(

            gdf[
                "CODSIA"
            ]
            .dropna()
            .unique()

        )


        # ====================================================
        # BUSCAR CODSIA
        # ====================================================
        #
        # Utilizamos isin() en lugar de buscar fila por fila.
        #
        # Esto es mucho más rápido.
        #
        # ====================================================

        mask_codsia = (

            df_capa[
                "CODSIA"
            ]

            .isin(
                codsias_shp
            )

        )


        # ----------------------------------------------------
        # CODSIA ENCONTRADO
        # ----------------------------------------------------

        indices_encontrados = (

            df_capa.index[
                mask_codsia
            ]

        )


        control.loc[
            indices_encontrados,
            "CODSIA_EN_SHP"
        ] = "SI"


        control.loc[
            indices_encontrados,
            "GEOMETRIA"
        ] = "SI"


        # ----------------------------------------------------
        # CODSIA NO ENCONTRADO
        # ----------------------------------------------------

        indices_no_encontrados = (

            df_capa.index[
                ~mask_codsia
            ]

        )


        control.loc[
            indices_no_encontrados,
            "ESTADO"
        ] = (
            "CODSIA_SIN_POLIGONO"
        )


        control.loc[
            indices_no_encontrados,
            "DETALLE"
        ] = (
            "CODSIA no encontrado "
            "en el SHP"
        )


        # ====================================================
        # PREPARAR DATOS PARA RASTER
        # ====================================================

        datos_raster = df_capa[
            [
                "CODSIA"
            ]
            +
            ATRIBUTOS
        ].copy()


        # ====================================================
        # RESOLVER DUPLICADOS DEL EXCEL
        # ====================================================
        #
        # Si un CODSIA aparece varias veces:
        #
        # - mismo valor -> se usa una vez
        #
        # - diferentes valores -> se toma el primer valor
        #   disponible y se registra el conflicto.
        #
        # ====================================================

        registros_raster = []


        for (

            codsia,

            grupo

        ) in datos_raster.groupby(

            "CODSIA",

            dropna=False

        ):


            # ------------------------------------------------
            # CODSIA vacío
            # ------------------------------------------------

            if pd.isna(
                codsia
            ):

                continue


            fila_raster = {

                "CODSIA":
                    codsia

            }


            conflicto = False


            # ------------------------------------------------
            # Cada atributo
            # ------------------------------------------------

            for atributo in ATRIBUTOS:

                valores = (

                    grupo[
                        atributo
                    ]

                    .dropna()

                    .unique()

                )


                # --------------------------------------------
                # Sin valor
                # --------------------------------------------

                if len(
                    valores
                ) == 0:

                    fila_raster[
                        atributo
                    ] = np.nan


                # --------------------------------------------
                # Con valor
                # --------------------------------------------

                else:

                    fila_raster[
                        atributo
                    ] = valores[0]


                    # ----------------------------------------
                    # Diferentes valores
                    # ----------------------------------------

                    if len(
                        valores
                    ) > 1:

                        conflicto = True


            registros_raster.append(

                fila_raster

            )


            # ------------------------------------------------
            # Registrar conflicto
            # ------------------------------------------------

            if conflicto:

                indices_conflicto = (

                    df_capa.index[

                        df_capa[
                            "CODSIA"
                        ]
                        ==
                        codsia

                    ]

                )


                control.loc[
                    indices_conflicto,
                    "ESTADO"
                ] = (
                    "DUPLICADO_CON_CONFLICTO"
                )


                control.loc[
                    indices_conflicto,
                    "DETALLE"
                ] = (

                    "CODSIA duplicado en "
                    "Excel con valores "
                    "diferentes; se utilizó "
                    "el primer valor para "
                    "rasterizar"

                )


        datos_raster = pd.DataFrame(

            registros_raster

        )


        # ====================================================
        # JOIN CON SHP
        # ====================================================

        gdf_resultado = gdf.merge(

            datos_raster,

            on="CODSIA",

            how="inner"

        )


        print(
            f"Polígonos relacionados: "
            f"{len(gdf_resultado):,}"
        )


        if gdf_resultado.empty:

            continue


        # ====================================================
        # ELIMINAR GEOMETRÍAS VACÍAS
        # ====================================================

        gdf_resultado = (

            gdf_resultado[

                gdf_resultado.geometry.notna()

            ]

            .copy()

        )


        if gdf_resultado.empty:

            continue


        # ====================================================
        # REPARAR GEOMETRÍAS INVÁLIDAS
        # ====================================================

        invalidas = (

            ~gdf_resultado.geometry.is_valid

        )


        if invalidas.any():

            print(

                f"Reparando "
                f"{invalidas.sum():,} "
                f"geometrías..."

            )


            try:

                gdf_resultado.loc[
                    invalidas,
                    "geometry"
                ] = (

                    gdf_resultado.loc[
                        invalidas,
                        "geometry"
                    ]
                    .make_valid()

                )

            except Exception as e:

                print(
                    f"Advertencia: {e}"
                )


        # ====================================================
        # CONSERVAR SOLO LO NECESARIO
        # ====================================================

        gdf_resultado = gdf_resultado[

            [
                "CODSIA",
                "geometry"
            ]
            +
            ATRIBUTOS

        ].copy()


        # ====================================================
        # AGREGAR
        # ====================================================

        lista_gdf.append(

            gdf_resultado

        )


    # ========================================================
    # NO HAY GEOMETRÍAS
    # ========================================================

    if not lista_gdf:

        print(

            f"\nNo hay geometrías "
            f"para {anio}."

        )

        continue


    # ========================================================
    # UNIR POLÍGONOS DEL AÑO
    # ========================================================

    gdf_anio = gpd.GeoDataFrame(

        pd.concat(

            lista_gdf,

            ignore_index=True

        ),

        crs=CRS_SALIDA

    )


    print(

        f"\nPolígonos totales "
        f"{anio}: "
        f"{len(gdf_anio):,}"

    )


    # ========================================================
    # EXTENSIÓN DEL RASTER
    # ========================================================

    minx, miny, maxx, maxy = (

        gdf_anio.total_bounds

    )


    # --------------------------------------------------------
    # Ajustar extensión al tamaño de píxel
    # --------------------------------------------------------

    minx = (

        np.floor(

            minx
            /
            RESOLUCION

        )

        *
        RESOLUCION

    )


    miny = (

        np.floor(

            miny
            /
            RESOLUCION

        )

        *
        RESOLUCION

    )


    maxx = (

        np.ceil(

            maxx
            /
            RESOLUCION

        )

        *
        RESOLUCION

    )


    maxy = (

        np.ceil(

            maxy
            /
            RESOLUCION

        )

        *
        RESOLUCION

    )


    # ========================================================
    # DIMENSIONES
    # ========================================================

    ancho = int(

        round(

            (
                maxx
                -
                minx
            )
            /
            RESOLUCION

        )

    )


    alto = int(

        round(

            (
                maxy
                -
                miny
            )
            /
            RESOLUCION

        )

    )


    print(

        f"Dimensiones: "
        f"{ancho:,} x {alto:,}"

    )


    # ========================================================
    # TRANSFORM
    # ========================================================

    transform = from_origin(

        minx,

        maxy,

        RESOLUCION,

        RESOLUCION

    )


    # ========================================================
    # RASTERIZAR CADA ATRIBUTO
    # ========================================================

    for atributo in ATRIBUTOS:

        print("\n")

        print(
            "=" * 70
        )

        print(
            f"RASTERIZANDO: "
            f"{atributo}"
        )

        print(
            "=" * 70
        )


        # ----------------------------------------------------
        # Preparar GeoDataFrame
        # ----------------------------------------------------

        gdf_raster = gdf_anio[

            [
                "CODSIA",
                "geometry",
                atributo
            ]

        ].copy()


        # ----------------------------------------------------
        # Convertir atributo
        # ----------------------------------------------------

        gdf_raster[
            "_VALOR"
        ] = convertir_a_numerico(

            gdf_raster[
                atributo
            ]

        )


        # ----------------------------------------------------
        # Eliminar valores sin información
        # ----------------------------------------------------

        gdf_raster = gdf_raster[

            gdf_raster[
                "_VALOR"
            ].notna()

        ].copy()


        if gdf_raster.empty:

            print(

                "No existen valores "
                "para rasterizar."

            )

            continue


        # ====================================================
        # RASTERIZACIÓN
        # ====================================================
        #
        # all_touched=True:
        #
        # Incluye los píxeles que tocan el polígono.
        #
        # Esto ayuda especialmente con lotes pequeños,
        # estrechos o irregulares.
        #
        # ====================================================

        shapes = zip(

            gdf_raster.geometry,

            gdf_raster[
                "_VALOR"
            ]

        )


        raster = rasterize(

            shapes=shapes,

            out_shape=(

                alto,

                ancho

            ),

            transform=transform,

            fill=NODATA,

            dtype="float32",

            all_touched=True

        )


        # ====================================================
        # NOMBRE DEL ATRIBUTO
        # ====================================================

        nombre_atributo = (

            limpiar_nombre_archivo(

                atributo

            )

        )


        # ====================================================
        # NOMBRE TIFF
        # ====================================================

        nombre_tiff = (

            f"{anio}_"
            f"{nombre_atributo}.tif"

        )


        ruta_tiff = os.path.join(

            CARPETA_SALIDA,

            nombre_tiff

        )


        # ====================================================
        # GUARDAR TIFF
        # ====================================================

        with rasterio.open(

            ruta_tiff,

            "w",

            driver="GTiff",

            height=alto,

            width=ancho,

            count=1,

            dtype="float32",

            crs=CRS_SALIDA,

            transform=transform,

            nodata=NODATA,

            compress="lzw",

            tiled=True,

            BIGTIFF="IF_SAFER"

        ) as dst:


            dst.write(

                raster,

                1

            )


            # ------------------------------------------------
            # Metadatos
            # ------------------------------------------------

            dst.update_tags(

                YEAR=str(
                    anio
                ),

                ATTRIBUTE=str(
                    atributo
                ),

                RESOLUTION=(

                    f"{RESOLUCION}x"
                    f"{RESOLUCION} m"

                ),

                CRS=CRS_SALIDA,

                SOURCE_EXCEL=(

                    os.path.basename(

                        ARCHIVO_EXCEL

                    )

                )

            )


        print(

            f"✓ Guardado: "
            f"{nombre_tiff}"

        )


        # ====================================================
        # RASTER DE IDENTIFICADORES
        # ====================================================
        #
        # Se utiliza solamente para saber cuántos píxeles
        # produjo cada CODSIA.
        #
        # NO se guarda en disco.
        #
        # ====================================================

        gdf_ids = gdf_raster[

            [
                "CODSIA",
                "geometry"
            ]

        ].copy()


        gdf_ids[
            "_ID_RASTER"
        ] = np.arange(

            1,

            len(gdf_ids) + 1

        )


        shapes_ids = zip(

            gdf_ids.geometry,

            gdf_ids[
                "_ID_RASTER"
            ]

        )


        raster_ids = rasterize(

            shapes=shapes_ids,

            out_shape=(

                alto,

                ancho

            ),

            transform=transform,

            fill=0,

            dtype="int32",

            all_touched=True

        )


        # ====================================================
        # COLUMNAS DEL CONTROL
        # ====================================================

        columna_valor = (

            f"VALOR_"
            f"{nombre_atributo}"

        )


        columna_pixel = (

            f"PIXEL_"
            f"{nombre_atributo}"

        )


        # ====================================================
        # ACTUALIZAR CONTROL
        # ====================================================

        for _, fila in gdf_ids.iterrows():

            codsia = fila[
                "CODSIA"
            ]


            id_raster = fila[
                "_ID_RASTER"
            ]


            cantidad_pixeles = int(

                (

                    raster_ids
                    ==
                    id_raster

                )

                .sum()

            )


            # ------------------------------------------------
            # Registros correspondientes en Excel
            # ------------------------------------------------

            indices_excel = (

                df_anio.index[

                    df_anio[
                        "CODSIA"
                    ]
                    ==
                    codsia

                ]

            )


            # ------------------------------------------------
            # Actualizar cada registro
            # ------------------------------------------------

            for idx in indices_excel:

                valor_excel = (

                    df_proceso.loc[
                        idx,
                        atributo
                    ]

                )


                # --------------------------------------------
                # El registro tiene valor
                # --------------------------------------------

                if not pd.isna(

                    valor_excel

                ):

                    control.loc[

                        idx,

                        columna_valor

                    ] = "SI"


                # --------------------------------------------
                # Cantidad de píxeles
                # --------------------------------------------

                control.loc[

                    idx,

                    columna_pixel

                ] = cantidad_pixeles


                # --------------------------------------------
                # Estado
                # --------------------------------------------

                if cantidad_pixeles > 0:

                    estado_actual = (

                        control.loc[
                            idx,
                            "ESTADO"
                        ]

                    )


                    # ----------------------------------------
                    # Rasterizado correctamente
                    # ----------------------------------------

                    if (

                        estado_actual
                        ==
                        "PENDIENTE"

                    ):

                        control.loc[
                            idx,
                            "ESTADO"
                        ] = (
                            "RASTERIZADO"
                        )


                        control.loc[
                            idx,
                            "DETALLE"
                        ] = (

                            f"{atributo}: "
                            f"rasterizado"

                        )


                    # ----------------------------------------
                    # Tenía conflicto
                    # ----------------------------------------

                    elif (

                        estado_actual
                        ==
                        "DUPLICADO_CON_CONFLICTO"

                    ):

                        control.loc[
                            idx,
                            "DETALLE"
                        ] = (

                            control.loc[
                                idx,
                                "DETALLE"
                            ]

                            +

                            f" | {atributo}: "
                            f"rasterizado"

                        )


                else:

                    control.loc[

                        idx,

                        "ESTADO"

                    ] = (

                        "SIN_PIXELES"

                    )


                    control.loc[

                        idx,

                        "DETALLE"

                    ] = (

                        f"El polígono existe "
                        f"pero no produjo "
                        f"píxeles para "
                        f"{atributo}"

                    )


    # ========================================================
    # COMPLETAR ESTADOS PENDIENTES
    # ========================================================

    for idx in df_anio.index:

        estado = control.loc[

            idx,

            "ESTADO"

        ]


        if estado == "PENDIENTE":

            # ------------------------------------------------
            # CODSIA encontrado pero no rasterizado
            # ------------------------------------------------

            if (

                control.loc[
                    idx,
                    "CODSIA_EN_SHP"
                ]
                ==
                "SI"

            ):

                control.loc[
                    idx,
                    "ESTADO"
                ] = (
                    "NO_RASTERIZADO"
                )


                control.loc[
                    idx,
                    "DETALLE"
                ] = (

                    "CODSIA y geometría "
                    "encontrados, pero "
                    "no fue incorporado "
                    "al raster"

                )


# ============================================================
# 17. GUARDAR ÚNICAMENTE CONTROL_EXCEL.CSV
# ============================================================
#
# NO SE GENERA:
#
#   RESUMEN_CONTROL.csv
#   RESUMEN_CONTROL_POR_ANO.csv
#
# SOLO:
#
#   CONTROL_EXCEL.csv
#
# ============================================================


print("\n")

print(
    "=" * 80
)

print(
    "GUARDANDO CONTROL_EXCEL.CSV"
)

print(
    "=" * 80
)


ruta_control = os.path.join(

    CARPETA_SALIDA,

    "CONTROL_EXCEL.csv"

)


# ------------------------------------------------------------
# Guardar solamente los registros procesados
# ------------------------------------------------------------

control.to_csv(

    ruta_control,

    index=False,

    encoding="utf-8-sig"

)


print(
    "\n✓ CONTROL_EXCEL.csv generado"
)


print(
    f"Registros en control: "
    f"{len(control):,}"
)


print(
    f"Archivo:"
)


print(
    ruta_control
)


# ============================================================
# 18. RESUMEN SOLO EN CONSOLA
# ============================================================
#
# NO SE GUARDA NINGÚN CSV ADICIONAL.
#
# ============================================================


print("\n")

print(
    "=" * 80
)

print(
    "RESUMEN DEL PROCESO"
)

print(
    "=" * 80
)


print(

    control[
        "ESTADO"
    ].value_counts(
        dropna=False
    )

)


# ============================================================
# 19. FINAL
# ============================================================


print("\n")

print(
    "=" * 80
)

print(
    "PROCESO TERMINADO"
)

print(
    "=" * 80
)


print(
    f"\nAños procesados: "
    f"{ANOS}"
)


print(
    f"Registros procesados: "
    f"{len(df_proceso):,}"
)


print(
    "\nCarpeta de resultados:"
)


print(
    CARPETA_SALIDA
)


print(
    "\nControl:"
)


print(
    ruta_control
)



GENERADOR DE RASTER HISTÓRICOS

Años seleccionados: [2024]
Atributos: ['Variedad_TIFF', 'Corte', 'Edad', 'Rdto', 'TAH', 'TAHM', '%Sac.Cana', 'Num.Riegos', 'Vejez', 'Semanas mad.', 'Luvias', 'Lluvias (0 - 3 Meses)', 'Lluvias 10d Ant.Cos', 'Ton.Azucar']
Resolución: 10 x 10 metros


LECTURA DEL EXCEL
Leyendo únicamente las columnas necesarias...
Filas leídas: 53,116


DATOS SELECCIONADOS
Años seleccionados: [2024]
Registros a procesar: 3,919



################################################################################
# PROCESANDO 2024
################################################################################
Registros: 3,919
SHP diferentes: 1


----------------------------------------------------------------------
Procesando: CP_2024_dp.shp
----------------------------------------------------------------------
C:\Users\jjruiz\OneDrive - Sector Agro\Geodata\Analisis datos a raster\TEMP\depurados\CP_2024_dp.shp
Polígonos SHP: 4,136
Polígonos relacionados: 4,097

Polígonos tot

KeyboardInterrupt: 

In [ ]:
# ============================================================
# GENERADOR RÁPIDO DE RASTER HISTÓRICOS DE CULTIVOS
# ============================================================
#
# FUENTES:
#
#   1. Excel histórico
#   2. Shapefiles históricos
#
# LLAVE:
#
#   CODSIA
#
# RESULTADOS:
#
#   2022_EDAD.tif
#   2022_AREA_BRUTA.tif
#   2024_EDAD.tif
#   2024_AREA_BRUTA.tif
#   ...
#
#
# IMPORTANTE:
#
# ESTA VERSIÓN NO GENERA NINGÚN INFORME.
#
# NO GENERA:
#
#   CONTROL_EXCEL.csv
#   RESUMEN_CONTROL.csv
#   RESUMEN_CONTROL_POR_ANO.csv
#
# Su objetivo es generar únicamente los TIFF
# de la manera más rápida posible.
#
#
# CODSIA:
#
#   - Siempre se trata como TEXTO.
#   - Nunca se convierte a número.
#   - Se conserva el 0 inicial.
#   - Se admiten:
#
#       010001008
#
#   y:
#
#       010001009B
#
# ============================================================


# ============================================================
# IMPORTACIONES
# ============================================================

import os
import re

import pandas as pd
import geopandas as gpd
import numpy as np

import rasterio

from rasterio.transform import from_origin

from rasterio.features import rasterize


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================


# ------------------------------------------------------------
# CARPETA PRINCIPAL
# ------------------------------------------------------------

CARPETA_DATOS = (
    r"C:\Users\jjruiz\OneDrive - Sector Agro\Geodata\Analisis datos a raster\TEMP\depurados"
)


# ------------------------------------------------------------
# ARCHIVO EXCEL
# ------------------------------------------------------------

ARCHIVO_EXCEL = os.path.join(
    CARPETA_DATOS,
    "Reporte_Biosal.xlsx"
)


# ------------------------------------------------------------
# CARPETA DE SALIDA
# ------------------------------------------------------------

CARPETA_SALIDA = os.path.join(
    CARPETA_DATOS,
    "RASTER_HISTORICOS"
)


os.makedirs(
    CARPETA_SALIDA,
    exist_ok=True
)


# ------------------------------------------------------------
# AÑOS A PROCESAR
#
# EJEMPLOS:
#
# ANOS = [2020]
#
# ANOS = [2022, 2024]
#
# ANOS = [2019, 2020, 2021, 2022]
#
# TODOS:
#
# ANOS = list(range(2014, 2026))
# ------------------------------------------------------------

#ANOS = [2014,2015,2016,2017,2018,2019,2020,2021,2022,2024,2025]
ANOS = [2026]


# ------------------------------------------------------------
# ATRIBUTOS A RASTERIZAR
#
# Deben existir exactamente con estos nombres en el Excel.
# ------------------------------------------------------------

ATRIBUTOS = ["Variedad_TIFF","Corte","Edad","Rdto","TAH","TAHM","%Sac.Cana","Num.Riegos","Vejez","Semanas mad.","Luvias","Lluvias (0 - 3 Meses)","Lluvias 10d Ant.Cos","Ton.Azucar"]



# ------------------------------------------------------------
# RESOLUCIÓN DEL RASTER
#
# 10 = 10 x 10 metros
# ------------------------------------------------------------

RESOLUCION = 10


# ------------------------------------------------------------
# CRS DE SALIDA
# ------------------------------------------------------------

CRS_SALIDA = "EPSG:21896"


# ------------------------------------------------------------
# VALOR NODATA
# ------------------------------------------------------------

NODATA = -9999


# ============================================================
# 2. NORMALIZAR CODSIA
# ============================================================
#
# CODSIA ES UNA LLAVE ALFANUMÉRICA.
#
# NUNCA debe tratarse como número.
#
# Ejemplos:
#
#   10001008
#       ->
#   010001008
#
#   010001008
#       ->
#   010001008
#
#   10001009B
#       ->
#   010001009B
#
#   010001009B
#       ->
#   010001009B
#
# ============================================================


def normalizar_codsia(valor):

    # --------------------------------------------------------
    # Valor vacío
    # --------------------------------------------------------

    if pd.isna(valor):

        return None


    # --------------------------------------------------------
    # Siempre convertir a texto
    # --------------------------------------------------------

    texto = str(
        valor
    ).strip()


    # --------------------------------------------------------
    # Eliminar espacios externos
    # --------------------------------------------------------

    texto = texto.strip()


    # --------------------------------------------------------
    # Caso generado por Excel:
    #
    # 10001008.0
    #
    # --------------------------------------------------------

    if texto.endswith(".0"):

        texto = texto[:-2]


    # ========================================================
    # CASO 1
    #
    # SOLO NÚMEROS
    #
    # ========================================================

    if texto.isdigit():

        # ----------------------------------------------------
        # Completar siempre a 9 dígitos.
        # ----------------------------------------------------

        texto = texto.zfill(
            9
        )


        return texto


    # ========================================================
    # CASO 2
    #
    # NÚMEROS + UNA LETRA AL FINAL
    #
    # ========================================================

    match = re.fullmatch(

        r"(\d+)([A-Za-z])",

        texto

    )


    if match:

        parte_numerica = match.group(
            1
        )

        letra = match.group(
            2
        )


        # ----------------------------------------------------
        # Completar la parte numérica a 9 dígitos.
        # ----------------------------------------------------

        parte_numerica = (

            parte_numerica.zfill(
                9
            )

        )


        # ----------------------------------------------------
        # Conservar letra.
        # ----------------------------------------------------

        return (

            parte_numerica
            +
            letra

        )


    # ========================================================
    # OTROS FORMATOS
    #
    # No modificar.
    # ========================================================

    return texto


# ============================================================
# 3. LIMPIAR NOMBRE DEL ATRIBUTO
# ============================================================


def limpiar_nombre_archivo(nombre):

    nombre = str(
        nombre
    ).strip()


    # --------------------------------------------------------
    # Espacios -> _
    # --------------------------------------------------------

    nombre = nombre.replace(
        " ",
        "_"
    )


    # --------------------------------------------------------
    # Eliminar caracteres especiales
    # --------------------------------------------------------

    nombre = re.sub(

        r"[^A-Za-z0-9_\-]",

        "",

        nombre

    )


    return nombre.upper()


# ============================================================
# 4. OBTENER RUTA DEL SHAPEFILE
# ============================================================


def obtener_ruta_shp(nombre):

    nombre = str(
        nombre
    ).strip()


    # --------------------------------------------------------
    # Agregar extensión
    # --------------------------------------------------------

    if not nombre.lower().endswith(
        ".shp"
    ):

        nombre += ".shp"


    return os.path.join(

        CARPETA_DATOS,

        nombre

    )


# ============================================================
# 5. CONVERTIR ATRIBUTO A NUMÉRICO
# ============================================================


def convertir_a_numerico(serie):

    # --------------------------------------------------------
    # Si ya es numérico
    # --------------------------------------------------------

    if pd.api.types.is_numeric_dtype(
        serie
    ):

        return pd.to_numeric(

            serie,

            errors="coerce"

        )


    # --------------------------------------------------------
    # Convertir temporalmente a texto
    # --------------------------------------------------------

    serie = (

        serie

        .astype("string")

        .str.strip()

    )


    # --------------------------------------------------------
    # Coma decimal -> punto
    # --------------------------------------------------------

    serie = serie.str.replace(

        ",",

        ".",

        regex=False

    )


    return pd.to_numeric(

        serie,

        errors="coerce"

    )


# ============================================================
# 6. INICIO
# ============================================================


print("\n")

print(
    "=" * 80
)

print(
    "GENERADOR RÁPIDO DE RASTER HISTÓRICOS"
)

print(
    "=" * 80
)


print(
    f"\nAños seleccionados: {ANOS}"
)


print(
    f"Atributos: {ATRIBUTOS}"
)


print(
    f"Resolución: "
    f"{RESOLUCION} x {RESOLUCION} metros"
)


# ============================================================
# 7. VALIDAR ARCHIVO EXCEL
# ============================================================


if not os.path.exists(

    ARCHIVO_EXCEL

):

    raise FileNotFoundError(

        f"\nNo existe el archivo:\n"
        f"{ARCHIVO_EXCEL}"

    )


# ============================================================
# 8. DEFINIR COLUMNAS NECESARIAS
# ============================================================
#
# NO cargamos todo el Excel.
#
# Solamente necesitamos:
#
#   CODSIA
#   year
#   Capa_producto_final
#   atributos
#
# ============================================================


COLUMNAS_EXCEL = [

    "CODSIA",

    "year",

    "Capa_producto_final"

] + ATRIBUTOS


# ------------------------------------------------------------
# Eliminar posibles columnas repetidas
# ------------------------------------------------------------

COLUMNAS_EXCEL = list(

    dict.fromkeys(

        COLUMNAS_EXCEL

    )

)


# ============================================================
# 9. LEER EXCEL
# ============================================================


print("\n")

print(
    "=" * 80
)

print(
    "LECTURA DEL EXCEL"
)

print(
    "=" * 80
)


print(
    "Leyendo únicamente las columnas necesarias..."
)


df = pd.read_excel(

    ARCHIVO_EXCEL,
    sheet_name="Datos_depurados_eda_quemar",

    usecols=COLUMNAS_EXCEL,

    dtype={

        "CODSIA":
            "string"

    }

)


print(

    f"Filas leídas: "
    f"{len(df):,}"

)


# ============================================================
# 10. VALIDAR COLUMNAS
# ============================================================


COLUMNAS_REQUERIDAS = [

    "CODSIA",

    "year",

    "Capa_producto_final"

] + ATRIBUTOS


faltantes = [

    columna

    for columna in COLUMNAS_REQUERIDAS

    if columna not in df.columns

]


if faltantes:

    raise ValueError(

        "\nFaltan columnas en el Excel:\n\n"

        +
        "\n".join(
            faltantes
        )

    )


# ============================================================
# 11. NORMALIZAR EXCEL
# ============================================================


# ------------------------------------------------------------
# CODSIA
# ------------------------------------------------------------

df["CODSIA"] = (

    df["CODSIA"]

    .apply(
        normalizar_codsia
    )

    .astype("string")

)


# ------------------------------------------------------------
# YEAR
# ------------------------------------------------------------

df["year"] = pd.to_numeric(

    df["year"],

    errors="coerce"

)


# ------------------------------------------------------------
# CAPA
# ------------------------------------------------------------

df["Capa_producto_final"] = (

    df[
        "Capa_producto_final"
    ]

    .astype("string")

    .str.strip()

)


# ============================================================
# 12. FILTRAR LOS AÑOS
# ============================================================
#
# Después de esta operación solamente trabajamos con los
# años solicitados.
#
# ============================================================


df_proceso = df[

    df["year"].isin(

        ANOS

    )

].copy()


# ------------------------------------------------------------
# Liberar el DataFrame completo.
# ------------------------------------------------------------

del df


# ============================================================
# 13. INFORMACIÓN DEL FILTRO
# ============================================================


print("\n")

print(
    "=" * 80
)

print(
    "DATOS SELECCIONADOS"
)

print(
    "=" * 80
)


print(

    f"Años seleccionados: "
    f"{ANOS}"

)


print(

    f"Registros a procesar: "
    f"{len(df_proceso):,}"

)


# ------------------------------------------------------------
# Si no hay registros
# ------------------------------------------------------------

if df_proceso.empty:

    print(

        "\nNo existen registros "
        "para los años seleccionados."

    )

    raise SystemExit


# ============================================================
# 14. PROCESAR AÑO POR AÑO
# ============================================================


for anio in ANOS:

    print("\n\n")

    print(
        "#" * 80
    )

    print(
        f"# PROCESANDO {anio}"
    )

    print(
        "#" * 80
    )


    # --------------------------------------------------------
    # Filtrar el año
    # --------------------------------------------------------

    df_anio = df_proceso[

        df_proceso["year"] == anio

    ].copy()


    print(

        f"Registros: "
        f"{len(df_anio):,}"

    )


    if df_anio.empty:

        print(
            "No existen registros."
        )

        continue


    # ========================================================
    # AGRUPAR POR SHAPEFILE
    # ========================================================

    grupos = df_anio.groupby(

        "Capa_producto_final",

        dropna=False

    )


    print(

        f"SHP diferentes: "
        f"{len(grupos):,}"

    )


    # ========================================================
    # LISTA DE GEODATAFRAMES
    # ========================================================

    lista_gdf = []


    # ========================================================
    # PROCESAR CADA SHP
    # ========================================================

    for (

        nombre_capa,

        df_capa

    ) in grupos:


        print("\n")

        print(
            "-" * 70
        )

        print(

            f"Procesando: "
            f"{nombre_capa}"

        )

        print(
            "-" * 70
        )


        # ====================================================
        # CAPA VACÍA
        # ====================================================

        if pd.isna(

            nombre_capa

        ):

            print(
                "⚠ Capa vacía"
            )

            continue


        # ====================================================
        # RUTA DEL SHP
        # ====================================================

        ruta_shp = obtener_ruta_shp(

            nombre_capa

        )


        print(
            ruta_shp
        )


        # ====================================================
        # SHP NO ENCONTRADO
        # ====================================================

        if not os.path.exists(

            ruta_shp

        ):

            print(
                "❌ SHP NO ENCONTRADO"
            )

            continue


        # ====================================================
        # LEER SHP
        # ====================================================

        try:

            gdf = gpd.read_file(

                ruta_shp

            )


        except Exception as e:

            print(

                f"❌ Error leyendo SHP: "
                f"{e}"

            )

            continue


        print(

            f"Polígonos SHP: "
            f"{len(gdf):,}"

        )


        # ====================================================
        # VALIDAR CODSIA
        # ====================================================

        if "CODSIA" not in gdf.columns:

            print(

                "❌ El SHP no contiene "
                "la columna CODSIA"

            )

            continue


        # ====================================================
        # NORMALIZAR CODSIA DEL SHP
        # ====================================================
        #
        # EXACTAMENTE LA MISMA REGLA DEL EXCEL.
        #
        # ====================================================

        gdf["CODSIA"] = (

            gdf["CODSIA"]

            .apply(
                normalizar_codsia
            )

            .astype("string")

        )


        # ====================================================
        # VALIDAR CRS
        # ====================================================

        if gdf.crs is None:

            print(

                "❌ El SHP no tiene CRS."

            )

            continue


        # ====================================================
        # REPROYECTAR
        # ====================================================

        if gdf.crs != CRS_SALIDA:

            gdf = gdf.to_crs(

                CRS_SALIDA

            )


        # ====================================================
        # FILTRAR EL SHP
        # ====================================================
        #
        # Solamente nos interesan los CODSIA que aparecen
        # en el Excel del año que estamos procesando.
        #
        # Esto evita cargar a la siguiente etapa polígonos
        # que no necesitamos.
        #
        # ====================================================

        codsias_excel = set(

            df_capa[
                "CODSIA"
            ]

            .dropna()

            .unique()

        )


        gdf = gdf[

            gdf["CODSIA"].isin(

                codsias_excel

            )

        ].copy()


        print(

            f"Polígonos relacionados: "
            f"{len(gdf):,}"

        )


        # ----------------------------------------------------
        # Si no hay polígonos relacionados
        # ----------------------------------------------------

        if gdf.empty:

            continue


        # ====================================================
        # PREPARAR DATOS DEL EXCEL
        # ====================================================

        datos_raster = df_capa[

            [
                "CODSIA"
            ]
            +
            ATRIBUTOS

        ].copy()


        # ====================================================
        # RESOLVER DUPLICADOS DEL EXCEL
        # ====================================================
        #
        # Si un CODSIA aparece varias veces, se consolida
        # en un solo registro para el raster.
        #
        # ====================================================

        registros_raster = []


        for (

            codsia,

            grupo

        ) in datos_raster.groupby(

            "CODSIA",

            dropna=False

        ):


            # ------------------------------------------------
            # CODSIA vacío
            # ------------------------------------------------

            if pd.isna(

                codsia

            ):

                continue


            fila_raster = {

                "CODSIA":
                    codsia

            }


            # ------------------------------------------------
            # Procesar cada atributo
            # ------------------------------------------------

            for atributo in ATRIBUTOS:

                valores = (

                    grupo[
                        atributo
                    ]

                    .dropna()

                    .unique()

                )


                # --------------------------------------------
                # Sin valor
                # --------------------------------------------

                if len(

                    valores

                ) == 0:

                    fila_raster[
                        atributo
                    ] = np.nan


                # --------------------------------------------
                # Con valor
                # --------------------------------------------

                else:

                    # ----------------------------------------
                    # Se mantiene el primer valor.
                    # ----------------------------------------

                    fila_raster[
                        atributo
                    ] = valores[0]


            registros_raster.append(

                fila_raster

            )


        datos_raster = pd.DataFrame(

            registros_raster

        )


        # ====================================================
        # JOIN SHP + EXCEL
        # ====================================================

        gdf_resultado = gdf.merge(

            datos_raster,

            on="CODSIA",

            how="inner"

        )


        print(

            f"Polígonos para raster: "
            f"{len(gdf_resultado):,}"

        )


        # ----------------------------------------------------
        # Si no hay información
        # ----------------------------------------------------

        if gdf_resultado.empty:

            continue


        # ====================================================
        # ELIMINAR GEOMETRÍAS VACÍAS
        # ====================================================

        gdf_resultado = (

            gdf_resultado[

                gdf_resultado.geometry.notna()

            ]

            .copy()

        )


        if gdf_resultado.empty:

            continue


        # ====================================================
        # REPARAR GEOMETRÍAS INVÁLIDAS
        # ====================================================

        invalidas = (

            ~gdf_resultado.geometry.is_valid

        )


        if invalidas.any():

            print(

                f"Reparando "
                f"{invalidas.sum():,} "
                f"geometrías..."

            )


            try:

                gdf_resultado.loc[

                    invalidas,

                    "geometry"

                ] = (

                    gdf_resultado.loc[

                        invalidas,

                        "geometry"

                    ]

                    .make_valid()

                )


            except Exception as e:

                print(

                    f"⚠ Advertencia: "
                    f"{e}"

                )


        # ====================================================
        # CONSERVAR SOLO LAS COLUMNAS NECESARIAS
        # ====================================================

        gdf_resultado = gdf_resultado[

            [
                "CODSIA",
                "geometry"
            ]
            +
            ATRIBUTOS

        ].copy()


        # ====================================================
        # AGREGAR A LA LISTA
        # ====================================================

        lista_gdf.append(

            gdf_resultado

        )


    # ========================================================
    # NO HAY GEOMETRÍAS
    # ========================================================

    if not lista_gdf:

        print(

            f"\nNo hay geometrías "
            f"para {anio}."

        )

        continue


    # ========================================================
    # UNIR TODOS LOS POLÍGONOS DEL AÑO
    # ========================================================

    gdf_anio = gpd.GeoDataFrame(

        pd.concat(

            lista_gdf,

            ignore_index=True

        ),

        crs=CRS_SALIDA

    )


    print(

        f"\nPolígonos totales "
        f"{anio}: "
        f"{len(gdf_anio):,}"

    )


    # ========================================================
    # EXTENSIÓN DEL RASTER
    # ========================================================

    minx, miny, maxx, maxy = (

        gdf_anio.total_bounds

    )


    # --------------------------------------------------------
    # Ajustar al tamaño del píxel
    # --------------------------------------------------------

    minx = (

        np.floor(

            minx
            /
            RESOLUCION

        )

        *
        RESOLUCION

    )


    miny = (

        np.floor(

            miny
            /
            RESOLUCION

        )

        *
        RESOLUCION

    )


    maxx = (

        np.ceil(

            maxx
            /
            RESOLUCION

        )

        *
        RESOLUCION

    )


    maxy = (

        np.ceil(

            maxy
            /
            RESOLUCION

        )

        *
        RESOLUCION

    )


    # ========================================================
    # DIMENSIONES
    # ========================================================

    ancho = int(

        round(

            (
                maxx
                -
                minx
            )
            /
            RESOLUCION

        )

    )


    alto = int(

        round(

            (
                maxy
                -
                miny
            )
            /
            RESOLUCION

        )

    )


    print(

        f"Dimensiones: "
        f"{ancho:,} x {alto:,}"

    )


    # ========================================================
    # TRANSFORM
    # ========================================================

    transform = from_origin(

        minx,

        maxy,

        RESOLUCION,

        RESOLUCION

    )


    # ========================================================
    # RASTERIZAR CADA ATRIBUTO
    # ========================================================

    for atributo in ATRIBUTOS:

        print("\n")

        print(
            "=" * 70
        )

        print(

            f"RASTERIZANDO: "
            f"{atributo}"

        )

        print(
            "=" * 70
        )


        # ----------------------------------------------------
        # Seleccionar columnas
        # ----------------------------------------------------

        gdf_raster = gdf_anio[

            [
                "CODSIA",
                "geometry",
                atributo
            ]

        ].copy()


        # ----------------------------------------------------
        # Convertir atributo a número
        #
        # IMPORTANTE:
        #
        # Esto solamente se aplica al atributo.
        #
        # CODSIA nunca entra aquí.
        #
        # ----------------------------------------------------

        gdf_raster[

            "_VALOR"

        ] = convertir_a_numerico(

            gdf_raster[
                atributo
            ]

        )


        # ----------------------------------------------------
        # Eliminar registros sin valor
        # ----------------------------------------------------

        gdf_raster = gdf_raster[

            gdf_raster[
                "_VALOR"
            ].notna()

        ].copy()


        if gdf_raster.empty:

            print(

                "No existen valores "
                "para rasterizar."

            )

            continue


        # ====================================================
        # PREPARAR GEOMETRÍAS
        # ====================================================

        shapes = zip(

            gdf_raster.geometry,

            gdf_raster[
                "_VALOR"
            ]

        )


        # ====================================================
        # RASTERIZAR
        # ====================================================
        #
        # all_touched=True
        #
        # Incluye los píxeles que tocan el polígono.
        #
        # Esto ayuda a conservar lotes pequeños o estrechos.
        #
        # ====================================================

        raster = rasterize(

            shapes=shapes,

            out_shape=(

                alto,

                ancho

            ),

            transform=transform,

            fill=NODATA,

            dtype="float32",

            all_touched=True

        )


        # ====================================================
        # NOMBRE DEL ATRIBUTO
        # ====================================================

        nombre_atributo = (

            limpiar_nombre_archivo(

                atributo

            )

        )


        # ====================================================
        # NOMBRE DEL TIFF
        # ====================================================

        nombre_tiff = (

            f"{anio}_"
            f"{nombre_atributo}.tif"

        )


        ruta_tiff = os.path.join(

            CARPETA_SALIDA,

            nombre_tiff

        )


        # ====================================================
        # GUARDAR TIFF
        # ====================================================

        with rasterio.open(

            ruta_tiff,

            "w",

            driver="GTiff",

            height=alto,

            width=ancho,

            count=1,

            dtype="float32",

            crs=CRS_SALIDA,

            transform=transform,

            nodata=NODATA,

            compress="lzw",

            tiled=True,

            BIGTIFF="IF_SAFER"

        ) as dst:


            dst.write(

                raster,

                1

            )


            # ------------------------------------------------
            # Metadatos del TIFF
            # ------------------------------------------------

            dst.update_tags(

                YEAR=str(

                    anio

                ),

                ATTRIBUTE=str(

                    atributo

                ),

                RESOLUTION=(

                    f"{RESOLUCION}x"
                    f"{RESOLUCION} m"

                ),

                CRS=CRS_SALIDA,

                SOURCE_EXCEL=(

                    os.path.basename(

                        ARCHIVO_EXCEL

                    )

                )

            )


        print(

            f"✓ Guardado: "
            f"{ruta_tiff}"

        )


    # ========================================================
    # LIBERAR MEMORIA DEL AÑO
    # ========================================================

    del gdf_anio

    del lista_gdf

    del df_anio


# ============================================================
# 15. FINAL
# ============================================================


print("\n")

print(
    "=" * 80
)

print(
    "PROCESO TERMINADO"
)

print(
    "=" * 80
)


print(

    f"\nAños procesados: "
    f"{ANOS}"

)


print(

    f"\nCarpeta de resultados:"
)


print(

    CARPETA_SALIDA

)


print(

    "\nÚnicamente se generaron los raster "
    "solicitados."

)



GENERADOR RÁPIDO DE RASTER HISTÓRICOS

Años seleccionados: [2025]
Atributos: ['Variedad_TIFF', 'Corte', 'Edad', 'Rdto', 'TAH', 'TAHM', '%Sac.Cana', 'Num.Riegos', 'Vejez', 'Semanas mad.', 'Luvias', 'Lluvias (0 - 3 Meses)', 'Lluvias 10d Ant.Cos', 'Ton.Azucar']
Resolución: 10 x 10 metros


LECTURA DEL EXCEL
Leyendo únicamente las columnas necesarias...
Filas leídas: 53,116


DATOS SELECCIONADOS
Años seleccionados: [2025]
Registros a procesar: 2,312



################################################################################
# PROCESANDO 2025
################################################################################
Registros: 2,312
SHP diferentes: 1


----------------------------------------------------------------------
Procesando: CP_2025_dp.shp
----------------------------------------------------------------------
C:\Users\jjruiz\OneDrive - Sector Agro\Geodata\Analisis datos a raster\TEMP\depurados\CP_2025_dp.shp
Polígonos SHP: 2,614
Polígonos relacionados: 2,513
Polígon